# Predictor Variable Construction for CosMX RNA smiDE Pipeline

## Instructions

- **Purpose**: Create spatial predictor variables and add them to the Seurat object's metadata,
  so that `COSMX_RNA_smiDE_ADANA.ipynb` can reference them as `de_var`.
- **Pre-requisite**: The Seurat object must already have cell type annotations and spatial coordinates.
- Helper functions are loaded from `helper_functions.R`.
- Each predictor type is its own section, toggled via `compute_*` flags.
- The updated Seurat object is saved at the end with all new metadata columns.

**Predictor types:**
1. Distance to nearest target cell (continuous)
2. Number of neighboring target cells (continuous)
3. Mean neighborhood expression of a gene (continuous)
4. Transformation / binning of continuous variables (categorical or continuous)
5. Anatomical features via DBSCAN (logical)
6. Combine / recode existing annotations (categorical)

Reference: https://nanostring-biostats.github.io/CosMx-Analysis-Scratch-Space/posts/vignette-basic-analysis-updated/03_DE.html#crafting-predictor-variables-for-de

## Imports

In [30]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(patchwork)
library(data.table)
library(Matrix)
library(FNN)           # distance to nearest neighbor
library(dbscan)        # anatomical features (DBSCAN)
library(splines)       # spline transformation
library(InSituCor)     # neighbor graph & neighbor_sum

source("helper_functions.R")

## Configuration

All parameters for this run are defined in the cell below.
Change values here, then re-run the notebook.

In [32]:
# ============================================================
# CONFIGURATION -- edit this cell only
# ============================================================

# --- Study & Paths ---
study_name    <- "TMA18"
seu_file_path <- file.path("../outputs", study_name, "seurat_objects",
                           "annotated_object_TMA18_louvain_final_hieratype.RDS")
out_dir       <- "../outputs"
ASSAY_NAME    <- "RNA"

# --- Column Names ---
sdimx_col     <- "x_slide_mm"
sdimy_col     <- "y_slide_mm"
celltype_col  <- "celltype"
cellid_col    <- "cell_ID_new"
region_col    <- "region"            # prevents cross-region neighbors

# --- Predictor 1: Distance to Nearest Target Cell ---
compute_dist_to_target <- TRUE
dist_target_celltype   <- "epithelial"    # cell type to measure distance to
dist_k                 <- 1          # k-th nearest neighbor (1 = closest)
dist_col_name          <- "dist_to_nearest_tumor"

# --- Predictor 2: Number of Neighboring Target Cells ---
compute_neighbor_count <- TRUE
ncount_target_celltype <- "epithelial"    # cell type to count
ncount_col_name        <- "neighboring_tumor_count"

# --- Predictor 3: Mean Neighborhood Expression ---
compute_neighbor_expr  <- FALSE
nexpr_gene             <- "CXCL14"   # gene to average across neighbors
nexpr_normalize        <- TRUE       # normalize by nCount_RNA before averaging
nexpr_col_name         <- "neighbor_mean_CXCL14"

# --- Neighbor Graph (shared by Predictors 2 & 3) ---
compute_neighbor_graph <- TRUE
neighbor_method        <- "knn"      # "knn" or "radius"
neighbor_k             <- 50         # for knn: number of nearest neighbors
neighbor_radius        <- 0.1        # for radius: distance threshold in mm

# --- Predictor 4: Transformation / Binning (global) ---
# Define one or more transformations as a list. Each entry specifies:
#   source_col     - column in metadata to transform
#   method         - "quantile_bin", "equal_bin", "log", "rank_norm", "spline"
#   n_bins         - number of quantile bins (for binning/spline methods)
#   exclude_value  - (optional) value to exclude before binning (e.g. 0 for
#                    distance, where 0 = target cell itself). NULL = no exclusion.
#   exclude_label  - label for the excluded group (required if exclude_value set)
#   bin_labels     - labels for the n_bins quantile bins (NULL for auto).
#                    Length must match n_bins (NOT n_bins + 1).
#                    Labels go LOW -> HIGH:
#                      distance:  low = close,  high = far
#                      neighbors: low = few,    high = many
#   out_col        - output column name in metadata
#
# When exclude_value is set, the output factor has n_bins + 1 levels:
#   exclude_label | bin_label_1 | bin_label_2 | ... | bin_label_n
# When exclude_value is NULL, the output factor has n_bins levels.
compute_transformations <- TRUE
transformations <- list(
    # Distance: exclude 0s (target cells themselves), bin the rest into 3
    list(source_col     = "dist_to_nearest_tumor",
         method         = "quantile_bin",
         n_bins         = 3,
         exclude_value  = 0,
         exclude_label  = paste0("Target_",dist_target_celltype),
         bin_labels     = c("Near tumor", "Middle", "Far from tumor"),
         out_col        = "dist_to_tumor_bins"),

    # Neighbor count: no exclusion, normal 3-bin
    list(source_col     = "neighboring_tumor_count",
         method         = "quantile_bin",
         n_bins         = 3,
         exclude_value  = NULL,
         exclude_label  = NULL,
         bin_labels     = c("Few neighbors", "Some neighbors", "Many neighbors"),
         out_col        = "neighboring_tumor_bins")
)

# --- Predictor 5: Anatomical Features (DBSCAN) ---
compute_dbscan         <- FALSE
dbscan_target_celltype <- "endothelial"
dbscan_eps             <- 0.05       # neighborhood radius (mm)
dbscan_minPts          <- 5          # minimum points for core point
dbscan_col_name        <- "in_endothelial_structure"

# --- Predictor 6: Combine/Recode Existing Annotations ---
compute_recode         <- FALSE
recode_source_col      <- "celltype"
recode_mapping         <- list(
    "immune"  = c("cd4_naive", "cd4_tcm", "cd4_tem", "cd4_th1", "cd4_th2",
                  "cd4_th17", "cd4_treg", "cd8_naive", "cd8_tcm", "cd8_tem",
                  "cd8_cytotoxic", "cd8_exhausted", "nk", "bcell", "plasma",
                  "monocyte", "macrophage", "dendritic", "neutrophil", "mast"),
    "stromal" = c("fibroblast", "endothelial", "smooth_muscle"),
    "epithelial" = c("epithelial")
)
recode_col_name        <- "celltype_broad"

# --- Visualization ---
plot_spatial_validation <- TRUE
ptsize                 <- 0.1

# ============================================================
# DERIVED -- do not edit below this line
# ============================================================
run_tag     <- paste0(study_name, "_predictors")
pred_dir    <- file.path(out_dir, study_name, "predictors")
cache_dir   <- file.path(pred_dir, "cache")
figures_dir <- file.path(pred_dir, "figures")

dir.create(cache_dir,   showWarnings = FALSE, recursive = TRUE)
dir.create(figures_dir, showWarnings = FALSE, recursive = TRUE)

# Track all new columns added
new_columns <- character(0)

cat("Run tag:    ", run_tag, "\n")
cat("Output dir: ", pred_dir, "\n")

Run tag:     TMA18_predictors 
Output dir:  ../outputs/TMA18/predictors 


In [33]:
# Snapshot of all config parameters for reproducibility
config <- list(
    study_name = study_name, ASSAY_NAME = ASSAY_NAME,
    celltype_col = celltype_col, cellid_col = cellid_col,
    sdimx_col = sdimx_col, sdimy_col = sdimy_col, region_col = region_col,
    dist_target_celltype = dist_target_celltype, dist_k = dist_k,
    ncount_target_celltype = ncount_target_celltype,
    nexpr_gene = nexpr_gene, nexpr_normalize = nexpr_normalize,
    neighbor_method = neighbor_method, neighbor_k = neighbor_k,
    neighbor_radius = neighbor_radius,
    transformations = transformations,
    dbscan_target_celltype = dbscan_target_celltype,
    dbscan_eps = dbscan_eps, dbscan_minPts = dbscan_minPts,
    run_tag = run_tag, timestamp = Sys.time()
)
saveRDS(config, file.path(pred_dir, paste0("predictor_config_", run_tag, ".RDS")))
message("Config saved.")

Config saved.



## Data Loading & Preparation

In [34]:
seu <- readRDS(seu_file_path)
DefaultAssay(seu) <- ASSAY_NAME
cat("Loaded:", ncol(seu), "cells x", nrow(seu), "genes\n")

# Extract counts: cells in rows, genes in columns
counts   <- Matrix::t(seu[[ASSAY_NAME]]$counts)
metadata <- seu@meta.data
xy       <- as.matrix(metadata[, c(sdimx_col, sdimy_col)])

# Alignment check
stopifnot(all(rownames(counts) == rownames(metadata)))
stopifnot(all(rownames(counts) == rownames(xy)))

cat("\nCell types:\n")
print(sort(table(metadata[[celltype_col]]), decreasing = TRUE))
cat("\nRegions:\n")
print(table(metadata[[region_col]]))

Loaded: 189704 cells x 6175 genes

Cell types:

   epithelial    fibroblast smooth_muscle   endothelial    macrophage 
        72661         24475         16344         14566         11823 
     monocyte        plasma         bcell    neutrophil       cd8_tem 
        10454          9793          3429          3095          2835 
    dendritic      cd4_treg cd8_cytotoxic            nk          mast 
         2574          2484          2315          1886          1833 
    cd8_naive cd8_exhausted       cd4_tem     cd4_naive       cd4_tcm 
         1294          1254          1191          1127          1104 
      cd4_th2      cd4_th17       cd4_th1       cd8_tcm 
          976           771           760           660 

Regions:

 18_A2  18_A3  18_A4  18_A5  18_A6  18_A7  18_A8  18_A9 18_A10  18_B2  18_B3 
  2406   3329   3591   2716   4338   2369   2427   2616   1466   2341   4984 
 18_B4  18_B5  18_B6  18_B7  18_B8  18_B9 18_B10  18_C2  18_C3  18_C4  18_C5 
  2398   2405   3802   22

## Neighbor Graph Construction

Builds a spatial neighbor graph used by Predictors 2 (neighbor count) and 3 (mean neighborhood expression).
The `region` column prevents cross-region neighbor links.

**Methods** (set `neighbor_method` in config):
- `"radius"` — all neighbors within `neighbor_radius` mm (`InSituCor:::radiusBasedGraph`)
- `"knn"` — k nearest neighbors (`InSituCor:::nearestNeighborGraph`)

Cached to avoid re-computation on kernel restart.

In [35]:
neighbor_graph_path <- file.path(cache_dir, paste0("neighbor_graph_", neighbor_method, ".RDS"))

if (compute_neighbor_graph) {
    message("Building neighbor graph (method: ", neighbor_method, ") ...")
    tissue <- metadata[[region_col]]

    if (neighbor_method == "radius") {
        neighbors <- InSituCor:::radiusBasedGraph(
            x = xy[, 1], y = xy[, 2],
            R = neighbor_radius,
            subset = tissue
        )
        cat("Method: radius-based (R =", neighbor_radius, "mm)\n")
    } else if (neighbor_method == "knn") {
        neighbors <- InSituCor:::nearestNeighborGraph(
            x = xy[, 1], y = xy[, 2],
            N = neighbor_k,
            subset = tissue
        )
        cat("Method: k-NN (k =", neighbor_k, ")\n")
    } else {
        stop("Unknown neighbor_method: '", neighbor_method,
             "'. Use 'radius' or 'knn'.")
    }

    stopifnot(identical(dim(neighbors), rep(nrow(metadata), 2)))

    saveRDS(neighbors, neighbor_graph_path)
    message("Saved to: ", neighbor_graph_path)
} else {
    message("Loading cached neighbor graph from: ", neighbor_graph_path)
    neighbors <- readRDS(neighbor_graph_path)
}

nn_per_cell <- Matrix::rowSums(neighbors > 0)
cat("Neighbor graph:", nrow(neighbors), "x", ncol(neighbors), "\n")
cat("Mean neighbors per cell:", round(mean(nn_per_cell), 1), "\n")
cat("Median neighbors per cell:", median(nn_per_cell), "\n")

Building neighbor graph (method: knn) ...



Method: k-NN (k = 50 )


Saved to: ../outputs/TMA18/predictors/cache/neighbor_graph_knn.RDS



Neighbor graph: 189704 x 189704 
Mean neighbors per cell: 50 
Median neighbors per cell: 50 


## Predictor 1: Distance to Nearest Target Cell

Computes the Euclidean distance from every cell to its k-th nearest cell of a target type.
Uses `FNN::get.knnx` for fast exact nearest-neighbor search.
Computed per-region to avoid cross-region artifacts.

**Output**: continuous column (distance in mm). Transform it (Section below) before using as `de_var` in smiDE.

In [36]:
if (compute_dist_to_target) {
    message("Computing distance to nearest '", dist_target_celltype, "' ...")

    is_target <- metadata[[celltype_col]] == dist_target_celltype
    cat("Target cells ('", dist_target_celltype, "'):", sum(is_target), "/", nrow(metadata), "\n")

    if (sum(is_target) < dist_k) {
        stop("Fewer target cells (", sum(is_target), ") than k (", dist_k,
             "). Cannot compute distance.")
    }

    # Per-region to avoid cross-region artifacts
    dist_vec <- rep(NA_real_, nrow(metadata))
    for (reg in unique(metadata[[region_col]])) {
        in_reg     <- metadata[[region_col]] == reg
        is_tgt_reg <- in_reg & is_target

        if (sum(is_tgt_reg) < dist_k) {
            warning("Region '", reg, "' has < ", dist_k, " target cells. Setting distance to NA.")
            next
        }

        dist_vec[in_reg] <- FNN::get.knnx(
            data  = as.matrix(xy[is_tgt_reg, , drop = FALSE]),
            query = as.matrix(xy[in_reg, , drop = FALSE]),
            k     = dist_k
        )$nn.dist[, dist_k]
    }

    metadata[[dist_col_name]] <- dist_vec
    new_columns <- c(new_columns, dist_col_name)

    cat("Column '", dist_col_name, "' added.\n")
    cat("  Range:", round(min(dist_vec, na.rm = TRUE), 4), "-",
        round(max(dist_vec, na.rm = TRUE), 4), "mm\n")
    cat("  Median:", round(median(dist_vec, na.rm = TRUE), 4), "mm\n")
    cat("  NAs:", sum(is.na(dist_vec)), "\n")
}

Computing distance to nearest 'epithelial' ...



Target cells (' epithelial '): 72661 / 189704 
Column ' dist_to_nearest_tumor ' added.
  Range: 0 - 0.455 mm
  Median: 0.0125 mm
  NAs: 0 


In [37]:
if (compute_dist_to_target && plot_spatial_validation) {
    p_spatial <- xyplot(
        cluster_column = dist_col_name,
        metadata = metadata,
        x_column = sdimx_col,
        y_column = sdimy_col,
        ptsize = ptsize,
        continuous_palette = function(n) colorRampPalette(c("darkred", "gold", "grey90"))(n)
    ) + labs(title = paste0("Predictor 1: ", dist_col_name), color = "Distance (mm)")

    p_hist <- ggplot(metadata, aes(x = .data[[dist_col_name]])) +
        geom_histogram(bins = 50, fill = "steelblue", color = "white") +
        theme_minimal() +
        labs(title = paste0("Distribution: ", dist_col_name),
             x = "Distance (mm)", y = "Count")

    # print(p_spatial + p_hist + plot_layout(widths = c(2, 1)))

    ggsave(file.path(figures_dir, paste0("pred1_", dist_col_name, ".pdf")),
           p_spatial + p_hist + plot_layout(widths = c(2, 1)),
           width = 14, height = 6)
}

## Predictor 2: Number of Neighboring Target Cells

Counts how many cells of a target type are in each cell's spatial neighborhood.
Uses `InSituCor:::neighbor_sum` with a binary indicator vector.

**Prerequisite**: Neighbor graph from section above.
**Output**: continuous column (integer-valued count).

In [38]:
if (compute_neighbor_count) {
    message("Computing neighbor count of '", ncount_target_celltype, "' ...")

    is_target <- metadata[[celltype_col]] == ncount_target_celltype
    cat("Target cells:", sum(is_target), "\n")

    ncount_vec <- InSituCor:::neighbor_sum(
        x = as.numeric(is_target),
        neighbors = neighbors
    )

    metadata[[ncount_col_name]] <- ncount_vec
    new_columns <- c(new_columns, ncount_col_name)

    cat("Column '", ncount_col_name, "' added.\n")
    cat("  Range:", min(ncount_vec), "-", max(ncount_vec), "\n")
    cat("  Mean:", round(mean(ncount_vec), 2), "\n")
    cat("  Cells with 0 target neighbors:", sum(ncount_vec == 0), "\n")
}

Computing neighbor count of 'epithelial' ...



Target cells: 72661 
Column ' neighboring_tumor_count ' added.
  Range: 0 - 50 
  Mean: 19.25 
  Cells with 0 target neighbors: 9680 


In [39]:
if (compute_neighbor_count && plot_spatial_validation) {
    p_spatial <- xyplot(
        cluster_column = ncount_col_name,
        metadata = metadata,
        x_column = sdimx_col,
        y_column = sdimy_col,
        ptsize = ptsize,
        continuous_palette = function(n) colorRampPalette(c("grey90", "#4B0082"))(n)
    ) + labs(title = paste0("Predictor 2: ", ncount_col_name), color = "# Neighbors")

    p_hist <- ggplot(metadata, aes(x = .data[[ncount_col_name]])) +
        geom_histogram(bins = 50, fill = "steelblue", color = "white") +
        theme_minimal() +
        labs(title = paste0("Distribution: ", ncount_col_name),
             x = "Neighbor count", y = "Count")

    # print(p_spatial + p_hist + plot_layout(widths = c(2, 1)))

    ggsave(file.path(figures_dir, paste0("pred2_", ncount_col_name, ".pdf")),
           p_spatial + p_hist + plot_layout(widths = c(2, 1)),
           width = 14, height = 6)
}

## Predictor 3: Mean Neighborhood Expression

Computes the average expression of a gene across each cell's neighbors.
Uses `InSituCor:::neighbor_sum` to sum expression, then divides by neighbor count for the mean.

**Prerequisite**: Neighbor graph from section above.
**Output**: continuous column.

In [40]:
if (compute_neighbor_expr) {
    message("Computing mean neighborhood expression of '", nexpr_gene, "' ...")

    if (!(nexpr_gene %in% colnames(counts))) {
        stop("Gene '", nexpr_gene, "' not found in count matrix.")
    }

    x <- as.numeric(counts[, nexpr_gene])
    if (nexpr_normalize) {
        x <- x / metadata$nCount_RNA
    }

    # Sum expression across neighbors
    neighbor_sum_vec <- InSituCor:::neighbor_sum(x = x, neighbors = neighbors)

    # Divide by neighbor count for true mean (avoid division by zero)
    n_neighbors <- Matrix::rowSums(neighbors > 0)
    neighbor_mean_vec <- ifelse(n_neighbors > 0,
                                neighbor_sum_vec / n_neighbors,
                                0)

    metadata[[nexpr_col_name]] <- neighbor_mean_vec
    new_columns <- c(new_columns, nexpr_col_name)

    cat("Column '", nexpr_col_name, "' added.\n")
    cat("  Range:", round(min(neighbor_mean_vec), 6), "-",
        round(max(neighbor_mean_vec), 6), "\n")
    cat("  Non-zero:", sum(neighbor_mean_vec > 0), "/", length(neighbor_mean_vec), "\n")
}

In [41]:
if (compute_neighbor_expr && plot_spatial_validation) {
    p_spatial <- xyplot(
        cluster_column = nexpr_col_name,
        metadata = metadata,
        x_column = sdimx_col,
        y_column = sdimy_col,
        ptsize = ptsize
    ) + labs(title = paste0("Predictor 3: ", nexpr_col_name), color = "Mean expr")

    p_hist <- ggplot(metadata, aes(x = .data[[nexpr_col_name]])) +
        geom_histogram(bins = 50, fill = "steelblue", color = "white") +
        theme_minimal() +
        labs(title = paste0("Distribution: ", nexpr_col_name),
             x = "Mean neighborhood expression", y = "Count")

    # print(p_spatial + p_hist + plot_layout(widths = c(2, 1)))

    ggsave(file.path(figures_dir, paste0("pred3_", nexpr_col_name, ".pdf")),
           p_spatial + p_hist + plot_layout(widths = c(2, 1)),
           width = 14, height = 6)
}

## Predictor 4: Transform Continuous Variables (Global)

Converts one or more continuous predictors into categorical or transformed variables.
Define multiple transformations in the `transformations` list in config — they all run in one pass.

**These are global bins across all cells.** For cell-type-specific binning
(e.g., binning within cd8_exhausted only), use the binning section in the smiDE notebook.

**Available methods:**
- `quantile_bin` — quantile-based binning, equal cell counts per bin **(recommended default)**
- `equal_bin` — equal-width binning
- `log` — log2(x + epsilon) transform (stays continuous)
- `rank_norm` — rank/quantile normalization to [0, 1] (stays continuous)
- `spline` — natural spline basis expansion (creates multiple columns `_s1`, `_s2`, ...)

**Value exclusion** (`exclude_value` / `exclude_label`):
Some variables have a special value that should not be mixed into continuous binning.
For example, `dist_to_nearest_tumor = 0` means the cell **is** the target type — that's
categorical, not a continuous distance. Set `exclude_value = 0` to pull these cells out
into their own group (labeled by `exclude_label`), then quantile-bin the rest into `n_bins` bins.
- `bin_labels` always has exactly `n_bins` entries (the quantile bins only)
- `exclude_label` is a separate label for the excluded group (first factor level)
- Set `exclude_value = NULL` when no exclusion is needed (normal binning)

**Label ordering**: `bin_labels` always go from **lowest to highest** value.
- Distance: low = close → `c("Near", ..., "Far")`
- Neighbor count: low = few → `c("Few", ..., "Many")`

In [42]:
if (compute_transformations) {
    for (ti in seq_along(transformations)) {
        tr <- transformations[[ti]]
        src_col    <- tr$source_col
        method     <- tr$method
        nbins      <- tr$n_bins
        labels     <- tr$bin_labels
        out_col    <- tr$out_col
        excl_val   <- tr$exclude_value
        excl_label <- tr$exclude_label

        cat("=== Transformation", ti, "/", length(transformations), "===\n")
        message("  '", src_col, "' -> '", out_col, "' (method: ", method, ")")

        x <- metadata[[src_col]]
        if (is.null(x)) stop("Source column '", src_col, "' not found in metadata.")

        if (method == "quantile_bin") {
            # --- Exclude specified value (if any) ---
            if (!is.null(excl_val)) {
                is_excl <- !is.na(x) & (x == excl_val)
                x_rest  <- x[!is_excl & !is.na(x)]
                cat("  Excluded", sum(is_excl), "cells with value", excl_val,
                    "(labeled '", excl_label, "')\n")
            } else {
                is_excl <- rep(FALSE, length(x))
                x_rest  <- x[!is.na(x)]
            }

            # --- Quantile-bin the remaining values ---
            quants   <- quantile(x_rest, probs = seq(0, 1, length.out = nbins + 1))
            uq       <- unique(quants)
            rest_cut <- cut(x_rest, breaks = uq, include.lowest = TRUE)

            if (length(uq) < nbins + 1) {
                warning("  Only ", length(levels(rest_cut)), " unique bins instead of ",
                        nbins, " (remaining values have ties).")
            }

            # Apply bin_labels to quantile bins
            if (!is.null(labels) && length(labels) == length(levels(rest_cut))) {
                levels(rest_cut) <- labels
            } else if (!is.null(labels) && length(labels) != length(levels(rest_cut))) {
                warning("  bin_labels length (", length(labels), ") != actual bins (",
                        length(levels(rest_cut)), "). Using auto labels.")
            }

            # --- Assemble full vector ---
            x_binned <- rep(NA_character_, length(x))
            if (!is.null(excl_val)) {
                x_binned[is_excl] <- excl_label
            }
            x_binned[!is_excl & !is.na(x)] <- as.character(rest_cut)

            # Factor with exclude label first (if applicable)
            if (!is.null(excl_val) && sum(is_excl) > 0) {
                lvls <- c(excl_label, levels(rest_cut))
            } else {
                lvls <- levels(rest_cut)
            }
            x_binned <- factor(x_binned, levels = lvls)

            metadata[[out_col]] <- x_binned
            new_columns <- c(new_columns, out_col)

            cat("  Quantile bins:\n")
            print(table(metadata[[out_col]], useNA = "ifany"))

        } else if (method == "equal_bin") {
            x_binned <- cut(x, breaks = nbins, include.lowest = TRUE)
            if (!is.null(labels) && length(labels) == length(levels(x_binned))) {
                levels(x_binned) <- labels
            }
            metadata[[out_col]] <- x_binned
            new_columns <- c(new_columns, out_col)

            cat("  Equal-width bins:\n")
            print(table(metadata[[out_col]], useNA = "ifany"))

        } else if (method == "log") {
            eps <- min(x[x > 0], na.rm = TRUE) * 0.1
            metadata[[out_col]] <- log2(x + eps)
            new_columns <- c(new_columns, out_col)

            cat("  Log2-transformed. Epsilon:", eps, "\n")
            cat("  Range:", range(metadata[[out_col]], na.rm = TRUE), "\n")

        } else if (method == "rank_norm") {
            metadata[[out_col]] <- rank(x, na.last = "keep") / sum(!is.na(x))
            new_columns <- c(new_columns, out_col)

            cat("  Rank-normalized to [0, 1].\n")

        } else if (method == "spline") {
            spline_mat <- as.data.frame(splines::ns(x, df = min(nbins, 20)))
            spline_cols <- paste0(out_col, "_s", seq_len(ncol(spline_mat)))
            colnames(spline_mat) <- spline_cols
            for (sc in spline_cols) {
                metadata[[sc]] <- spline_mat[[sc]]
            }
            new_columns <- c(new_columns, spline_cols)

            cat("  Spline basis expansion:", ncol(spline_mat), "columns added.\n")
            cat("  Column names:", paste(spline_cols, collapse = ", "), "\n")

        } else {
            stop("Unknown method: ", method)
        }
        cat("\n")
    }
}

=== Transformation 1 / 2 ===


  'dist_to_nearest_tumor' -> 'dist_to_tumor_bins' (method: quantile_bin)



  Excluded 72661 cells with value 0 (labeled ' Target_epithelial ')
  Quantile bins:

Target_epithelial        Near tumor            Middle    Far from tumor 
            72661             39018             39011             39014 

=== Transformation 2 / 2 ===


  'neighboring_tumor_count' -> 'neighboring_tumor_bins' (method: quantile_bin)



  Quantile bins:

 Few neighbors Some neighbors Many neighbors 
         65937          60684          63083 



In [46]:
if (compute_transformations && plot_spatial_validation) {
    for (ti in seq_along(transformations)) {
        tr <- transformations[[ti]]
        col_to_plot <- tr$out_col
        if (tr$method == "spline") {
            col_to_plot <- paste0(tr$out_col, "_s1")
        }

        if (is.factor(metadata[[col_to_plot]]) || is.character(metadata[[col_to_plot]])) {
            p_spatial <- xyplot(
                cluster_column = col_to_plot,
                metadata = metadata,
                x_column = sdimx_col,
                y_column = sdimy_col,
                ptsize = ptsize
            ) + labs(title = paste0("Pred 4.", ti, ": ", col_to_plot))

            p_bar <- ggplot(metadata, aes(x = .data[[col_to_plot]])) +
                geom_bar(fill = "steelblue", color = "white") +
                theme_minimal() +
                labs(title = paste0("Distribution: ", col_to_plot),
                     x = col_to_plot, y = "Count") +
                theme(axis.text.x = element_text(angle = 45, hjust = 1))
        } else {
            p_spatial <- xyplot(
                cluster_column = col_to_plot,
                metadata = metadata,
                x_column = sdimx_col,
                y_column = sdimy_col,
                ptsize = ptsize,
                continuous_palette = function(n) colorRampPalette(c("grey80", "#4B0082"))(n)
            ) + labs(title = paste0("Pred 4.", ti, ": ", col_to_plot))

            p_bar <- ggplot(metadata, aes(x = .data[[col_to_plot]])) +
                geom_histogram(bins = 50, fill = "steelblue", color = "white") +
                theme_minimal() +
                labs(title = paste0("Distribution: ", col_to_plot),
                     x = col_to_plot, y = "Count")
        }

        # print(p_spatial + p_bar + plot_layout(widths = c(2, 1)))

        ggsave(file.path(figures_dir, paste0("pred4_", col_to_plot, ".pdf")),
               p_spatial + p_bar + plot_layout(widths = c(2, 1)),
               width = 14, height = 6)
    }
}

## Predictor 5: Anatomical Features (DBSCAN)

Identifies contiguous spatial structures formed by a target cell type using DBSCAN clustering.
Cells belonging to a cluster (non-noise points) are labeled `TRUE`; all other cells are `FALSE`.
Computed per-region to avoid cross-region clusters.

**Output**: logical column (TRUE/FALSE).

In [44]:
if (compute_dbscan) {
    message("Running DBSCAN on '", dbscan_target_celltype, "' cells ...")

    is_target <- metadata[[celltype_col]] == dbscan_target_celltype
    cat("Target cells:", sum(is_target), "\n")

    if (sum(is_target) < dbscan_minPts) {
        warning("Fewer target cells than minPts. Skipping DBSCAN.")
    } else {
        metadata[[dbscan_col_name]] <- FALSE

        for (reg in unique(metadata[[region_col]])) {
            in_reg     <- metadata[[region_col]] == reg
            is_tgt_reg <- in_reg & is_target

            if (sum(is_tgt_reg) < dbscan_minPts) {
                cat("  Region '", reg, "': too few target cells, skipping.\n")
                next
            }

            db_result <- dbscan::dbscan(
                xy[is_tgt_reg, , drop = FALSE],
                eps    = dbscan_eps,
                minPts = dbscan_minPts
            )

            selected <- db_result$cluster != 0
            target_indices <- which(is_tgt_reg)
            metadata[[dbscan_col_name]][target_indices[selected]] <- TRUE

            cat("  Region '", reg, "':", sum(selected), "/", sum(is_tgt_reg),
                "cells in", max(db_result$cluster), "clusters\n")
        }

        new_columns <- c(new_columns, dbscan_col_name)
        cat("\nColumn '", dbscan_col_name, "' added.\n")
        print(table(metadata[[dbscan_col_name]]))
    }
}

In [45]:
if (compute_dbscan && plot_spatial_validation && dbscan_col_name %in% colnames(metadata)) {
    plot_col <- paste0(dbscan_col_name, "_chr")
    metadata[[plot_col]] <- as.character(metadata[[dbscan_col_name]])

    p_spatial <- xyplot(
        cluster_column = plot_col,
        metadata = metadata,
        x_column = sdimx_col,
        y_column = sdimy_col,
        cls = c("TRUE" = "red", "FALSE" = "grey85"),
        ptsize = ptsize
    ) + labs(title = paste0("Predictor 5: ", dbscan_col_name,
                            " (", dbscan_target_celltype, " structures)"))

    print(p_spatial)

    metadata[[plot_col]] <- NULL  # clean up temp column

    ggsave(file.path(figures_dir, paste0("pred5_", dbscan_col_name, ".pdf")),
           p_spatial, width = 10, height = 8)
}

## Predictor 6: Combine / Recode Existing Annotations

Creates a new metadata column by merging or recoding values from an existing column.
Useful for creating broad cell type categories (e.g., immune / stromal / epithelial).

In [47]:
if (compute_recode) {
    message("Recoding '", recode_source_col, "' into '", recode_col_name, "' ...")

    source_vals <- metadata[[recode_source_col]]
    new_vals <- rep(NA_character_, length(source_vals))

    for (new_label in names(recode_mapping)) {
        old_labels <- recode_mapping[[new_label]]
        new_vals[source_vals %in% old_labels] <- new_label
    }

    unmapped <- sum(is.na(new_vals))
    if (unmapped > 0) {
        unmapped_vals <- unique(source_vals[is.na(new_vals)])
        warning(unmapped, " cells could not be mapped. Unmapped values: ",
                paste(unmapped_vals, collapse = ", "))
        new_vals[is.na(new_vals)] <- "unmapped"
    }

    metadata[[recode_col_name]] <- factor(new_vals)
    new_columns <- c(new_columns, recode_col_name)

    cat("Column '", recode_col_name, "' added.\n")
    print(table(metadata[[recode_col_name]], useNA = "ifany"))
}

In [48]:
if (compute_recode && plot_spatial_validation) {
    p_spatial <- xyplot(
        cluster_column = recode_col_name,
        metadata = metadata,
        x_column = sdimx_col,
        y_column = sdimy_col,
        ptsize = ptsize
    ) + labs(title = paste0("Predictor 6: ", recode_col_name))

    p_bar <- ggplot(metadata, aes(x = .data[[recode_col_name]])) +
        geom_bar(fill = "steelblue", color = "white") +
        theme_minimal() +
        labs(title = paste0("Distribution: ", recode_col_name)) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1))

    # print(p_spatial + p_bar + plot_layout(widths = c(2, 1)))

    ggsave(file.path(figures_dir, paste0("pred6_", recode_col_name, ".pdf")),
           p_spatial + p_bar + plot_layout(widths = c(2, 1)),
           width = 14, height = 6)
}

## Summary & Save

Summarizes all new predictor columns, updates the Seurat object metadata, and saves.

In [49]:
cat("=== PREDICTOR SUMMARY ===", "\n\n")
cat("New columns added:", length(new_columns), "\n\n")

for (col in new_columns) {
    cat("--- ", col, " ---\n")
    vals <- metadata[[col]]

    if (is.factor(vals) || is.character(vals)) {
        cat("  Type: categorical (", length(unique(vals)), "levels)\n")
        print(table(vals, useNA = "ifany"))
    } else if (is.logical(vals)) {
        cat("  Type: logical\n")
        print(table(vals, useNA = "ifany"))
    } else if (is.numeric(vals)) {
        cat("  Type: numeric\n")
        cat("  Range:", round(range(vals, na.rm = TRUE), 4), "\n")
        cat("  Mean:", round(mean(vals, na.rm = TRUE), 4), "\n")
        cat("  Median:", round(median(vals, na.rm = TRUE), 4), "\n")
        cat("  NAs:", sum(is.na(vals)), "\n")
    }
    cat("\n")
}

# Identify columns ready for use as de_var in smiDE
suitable_de_vars <- new_columns[sapply(new_columns, function(col) {
    is.factor(metadata[[col]]) || is.character(metadata[[col]])
})]

if (length(suitable_de_vars) > 0) {
    cat("\n=== Columns ready for use as de_var in smiDE ===", "\n")
    for (col in suitable_de_vars) {
        cat("  ", col, ":", paste(levels(factor(metadata[[col]])), collapse = " | "), "\n")
    }
} else {
    cat("\nNo categorical columns created. Transform a continuous predictor",
        "(Predictor 4 section) to create a de_var suitable for smiDE.\n")
}

=== PREDICTOR SUMMARY === 

New columns added: 4 

---  dist_to_nearest_tumor  ---
  Type: numeric
  Range: 0 0.455 
  Mean: 0.0197 
  Median: 0.0125 
  NAs: 0 

---  neighboring_tumor_count  ---
  Type: numeric
  Range: 0 50 
  Mean: 19.247 
  Median: 16 
  NAs: 0 

---  dist_to_tumor_bins  ---
  Type: categorical ( 4 levels)
vals
Target_epithelial        Near tumor            Middle    Far from tumor 
            72661             39018             39011             39014 

---  neighboring_tumor_bins  ---
  Type: categorical ( 3 levels)
vals
 Few neighbors Some neighbors Many neighbors 
         65937          60684          63083 


=== Columns ready for use as de_var in smiDE === 
   dist_to_tumor_bins : Target_epithelial | Near tumor | Middle | Far from tumor 
   neighboring_tumor_bins : Few neighbors | Some neighbors | Many neighbors 


In [50]:
# Write all new columns back to Seurat metadata
for (col in new_columns) {
    seu@meta.data[[col]] <- metadata[[col]]
}

# Save updated object alongside input
seu_out_name <- sub("\\.RDS$", "_predictors.RDS", basename(seu_file_path))
seu_out_path <- file.path(dirname(seu_file_path), seu_out_name)

saveRDS(seu, seu_out_path)
cat("Updated Seurat object saved to:\n  ", seu_out_path, "\n")
cat("New columns:", paste(new_columns, collapse = ", "), "\n")

Updated Seurat object saved to:
   ../outputs/TMA18/seurat_objects/annotated_object_TMA18_louvain_final_hieratype_predictors.RDS 
New columns: dist_to_nearest_tumor, neighboring_tumor_count, dist_to_tumor_bins, neighboring_tumor_bins 


## Session Info

In [51]:
sessionInfo()

R version 4.5.2 (2025-10-31)
Platform: x86_64-apple-darwin13.4.0
Running under: macOS Tahoe 26.2

Matrix products: default
BLAS/LAPACK: /Users/fs2829/miniforge3/envs/COSMX_RNA_env_proj_ADANA/lib/libopenblasp-r0.3.30.dylib;  LAPACK version 3.12.0

locale:
[1] C/C.UTF-8/C/C/C/C

time zone: America/New_York
tzcode source: system (macOS)

attached base packages:
[1] splines   stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] InSituCor_0.0.0.9000 dbscan_1.2.4         FNN_1.1.4.1         
 [4] Matrix_1.7-4         data.table_1.17.8    patchwork_1.3.2     
 [7] ggplot2_4.0.1        dplyr_1.1.4          Seurat_5.4.0        
[10] SeuratObject_5.3.0   sp_2.2-0            

loaded via a namespace (and not attached):
  [1] RColorBrewer_1.1-3     jsonlite_2.0.0         magrittr_2.0.4        
  [4] spatstat.utils_3.2-1   farver_2.1.2           ragg_1.5.0            
  [7] vctrs_0.6.5            ROCR_1.0-11            Cairo_1.6-5           
 [10]

In [53]:
colnames(seu@meta.data)

[1] "fov"                                                                                                          
  [2] "Area"                                                                                                         
  [3] "AspectRatio"                                                                                                  
  [4] "x_FOV_px"                                                                                                     
  [5] "y_FOV_px"                                                                                                     
  [6] "Width"                                                                                                        
  [7] "Height"                                                                                                       
  [8] "Mean.PanCK"                                                                                                   
  [9] "Max.PanCK"                                                                                                    
 [10] "Mean.G"                                                                                                       
 [11] "Max.G"                                                                                                        
 [12] "Mean.Membrane"                                                                                                
 [13] "Max.Membrane"                                                                                                 
 [14] "Mean.CD45"                                                                                                    
 [15] "Max.CD45"                                                                                                     
 [16] "Mean.DAPI"                                                                                                    
 [17] "Max.DAPI"                                                                                                     
 [18] "SplitRatioToLocal"                                                                                            
 [19] "NucArea"                                                                                                      
 [20] "NucAspectRatio"                                                                                               
 [21] "Circularity"                                                                                                  
 [22] "Eccentricity"                                                                                                 
 [23] "Perimeter"                                                                                                    
 [24] "Solidity"                                                                                                     
 [25] "cell_id"                                                                                                      
 [26] "assay_type"                                                                                                   
 [27] "version"                                                                                                      
 [28] "Run_Tissue_name"                                                                                              
 [29] "Panel"                                                                                                        
 [30] "cellSegmentationSetId"                                                                                        
 [31] "cellSegmentationSetName"                                                                                      
 [32] "slide_ID_numeric"                                                                                             
 [33] "x_slide_mm"                                                                                                   
 [34] "y_slide_mm"                                                                                          